# NB09 — Verifikasi Galeri Ekstraksi Wajah

**Tujuan:** memastikan seluruh wajah yang terdeteksi InsightFace di NB01 benar-benar
wajah asli (bukan false-positive deteksi), termasuk wajah yang berakhir sebagai
**noise** di HDBSCAN (tidak masuk cluster manapun) — supaya jelas apakah wajah itu
gagal *dicluster* atau gagal *diekstrak*.

**Tiga cara browsing yang disediakan:**
1. **Semua wajah** (15.248) — paginasi, urut per `global_index`
2. **Khusus wajah NOISE** — paginasi, hanya wajah dengan `cluster_id == -1`
3. **Satu wajah representatif (medoid) per cluster** — sanity-check cepat semua cluster

**Input:** `output_nb05/metadata_labeled.pkl`, `output_nb01/embeddings.npy`,
`output_labeling/face_labels_verified.csv` (opsional, untuk anotasi status)

**Cara pakai:** ubah variabel `PAGE` di Cell 7/8, lalu jalankan ulang cell untuk
maju ke halaman berikutnya. Tidak perlu GPU.

In [ ]:
# Cell 1 — Imports & Mount
import pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

In [ ]:
# Cell 2 — Config
BASE       = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings")
NB01_DIR   = BASE / "output_nb01"
NB05_DIR   = BASE / "output_nb05"
LABEL_DIR  = BASE / "output_labeling"

THUMB_SIZE = 112     # ukuran thumbnail crop wajah
PADDING    = 0.15    # padding di sekitar bbox agar konteks wajah terlihat

print('Config OK.')

In [ ]:
# Cell 3 — Load Data
with open(NB05_DIR / "metadata_labeled.pkl", "rb") as f:
    meta = pickle.load(f)
N = len(meta)

labels = np.array([m["cluster_id"] for m in meta])   # -1 = noise

# Ground-truth opsional — untuk anotasi status (ok/noise/discard/split) di bawah thumbnail
gt_path = LABEL_DIR / "face_labels_verified.csv"
df_gt = pd.read_csv(gt_path) if gt_path.exists() else None
status_map = {}
if df_gt is not None:
    status_map = dict(zip(df_gt["global_index"], df_gt["status"]))

print(f"Total wajah   : {N:,}")
print(f"Noise         : {(labels == -1).sum():,} ({(labels == -1).mean()*100:.1f}%)")
print(f"Ter-cluster   : {(labels >= 0).sum():,}")
print(f"Jumlah cluster: {labels.max() + 1 if labels.max() >= 0 else 0}")
print(f"Ground-truth  : {'tersedia' if df_gt is not None else 'TIDAK ditemukan (anotasi status dilewati)'}")

In [ ]:
# Cell 4 — Helper: crop wajah dari foto asli (native resolution, dengan padding)
_img_cache = {}

def crop_face(gi, use_cache=True):
    """Crop wajah global_index gi dari foto asli, resize ke THUMB_SIZE persegi.
    Return None jika foto tidak terbaca atau bbox tidak valid.
    """
    m = meta[gi]
    photo_path = m["photo_path"]

    if use_cache:
        img = _img_cache.get(photo_path)
        if img is None:
            img = cv2.imread(str(photo_path))
            _img_cache[photo_path] = img
    else:
        img = cv2.imread(str(photo_path))

    if img is None:
        return None

    h, w = img.shape[:2]
    x1, y1, x2, y2 = m["bbox"]
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - bw * PADDING))
    y1 = max(0, int(y1 - bh * PADDING))
    x2 = min(w, int(x2 + bw * PADDING))
    y2 = min(h, int(y2 + bh * PADDING))

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    crop = cv2.resize(crop, (THUMB_SIZE, THUMB_SIZE))
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


def clear_cache():
    """Kosongkan cache foto — panggil kalau RAM mulai penuh saat browsing lama."""
    _img_cache.clear()


print('crop_face() siap. Panggil clear_cache() kalau RAM penuh.')

In [ ]:
# Cell 5 — Helper: galeri paginasi
def show_gallery(indices, page=0, per_page=100, cols=10, title=""):
    """Tampilkan grid thumbnail wajah untuk satu halaman dari daftar global_index.
    Label di bawah tiap thumbnail: global_index, cluster_id (-1=noise), status (jika ada).
    """
    total = len(indices)
    n_pages = (total + per_page - 1) // per_page
    page = max(0, min(page, n_pages - 1)) if n_pages > 0 else 0

    start = page * per_page
    end   = min(start + per_page, total)
    page_indices = indices[start:end]

    rows = (len(page_indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9))
    axes = np.atleast_2d(axes).flatten()

    for ax, gi in zip(axes, page_indices):
        crop = crop_face(gi)
        cid = labels[gi]
        status = status_map.get(gi, "")
        label = f"#{gi}\ncid={cid}"
        if status:
            label += f"\n{status}"
        if crop is None:
            ax.text(0.5, 0.5, "GAGAL\nBACA", ha='center', va='center', fontsize=8, color='red')
        else:
            ax.imshow(crop)
        ax.set_title(label, fontsize=6.5)
        ax.axis('off')

    for ax in axes[len(page_indices):]:
        ax.axis('off')

    fig.suptitle(f"{title}  —  Halaman {page+1}/{n_pages}  ({start+1}-{end} dari {total})",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('show_gallery() siap.')

In [ ]:
# Cell 6 — Ringkasan per cluster (termasuk noise)
sizes = Counter(labels.tolist())
rows = []
for cid, cnt in sorted(sizes.items(), key=lambda x: (x[0] != -1, x[0])):
    label_name = "NOISE (tidak masuk cluster)" if cid == -1 else f"Cluster {cid}"
    rows.append({"cluster_id": cid, "label": label_name, "jumlah_wajah": cnt})
df_summary = pd.DataFrame(rows)
print(f"Total baris ringkasan: {len(df_summary)} (1 baris noise + {len(df_summary)-1} cluster)")
display(df_summary.head(15))
print("...")
display(df_summary[df_summary['cluster_id'] == -1])

In [ ]:
# Cell 7 — BROWSE SEMUA WAJAH (15.248, urut global_index)
# Ubah PAGE untuk maju/mundur, lalu jalankan ulang cell ini.
PAGE     = 0      # <- ubah nomor halaman di sini (mulai dari 0)
PER_PAGE = 100
COLS     = 10

all_indices = list(range(N))
show_gallery(all_indices, page=PAGE, per_page=PER_PAGE, cols=COLS,
             title="SEMUA WAJAH (urut global_index)")
print(f"Total halaman: {(N + PER_PAGE - 1) // PER_PAGE}")

In [ ]:
# Cell 8 — BROWSE KHUSUS WAJAH NOISE (yang tidak masuk cluster manapun)
# Ini yang paling relevan untuk verifikasi: pastikan wajah noise BENAR wajah asli,
# bukan gagal ekstraksi/false-positive deteksi.
PAGE_NOISE     = 0    # <- ubah nomor halaman di sini
PER_PAGE_NOISE = 100
COLS_NOISE     = 10

noise_indices = np.where(labels == -1)[0].tolist()
print(f"Total wajah noise: {len(noise_indices):,}")
show_gallery(noise_indices, page=PAGE_NOISE, per_page=PER_PAGE_NOISE, cols=COLS_NOISE,
             title="WAJAH NOISE (tidak masuk cluster manapun)")
print(f"Total halaman: {(len(noise_indices) + PER_PAGE_NOISE - 1) // PER_PAGE_NOISE}")

In [ ]:
# Cell 9 — SATU WAJAH REPRESENTATIF (medoid) PER CLUSTER — sanity check cepat
# Medoid = wajah dalam cluster yang embedding-nya paling dekat ke rata-rata cluster.
embeddings = np.load(NB01_DIR / "embeddings.npy").astype(np.float32)
assert len(embeddings) == N, f"Mismatch: {len(embeddings)} embeddings vs {N} meta"

def medoid_index(idxs, emb):
    sub = emb[idxs]
    center = sub.mean(axis=0)
    dist = np.linalg.norm(sub - center, axis=1)
    return idxs[np.argmin(dist)]

cluster_to_indices = defaultdict(list)
for gi, cid in enumerate(labels):
    if cid >= 0:
        cluster_to_indices[cid].append(gi)

medoid_indices = []
for cid in sorted(cluster_to_indices.keys()):
    idxs = np.array(cluster_to_indices[cid])
    medoid_indices.append(int(medoid_index(idxs, embeddings)))

print(f"Jumlah cluster: {len(medoid_indices)}")
show_gallery(medoid_indices, page=0, per_page=len(medoid_indices), cols=10,
             title="1 WAJAH REPRESENTATIF (medoid) PER CLUSTER")